In [1]:
from google.colab import files
import zipfile
import os

print("Selecione o arquivo 'size_normalized_texts.zip' do seu computador:")
uploaded = files.upload()

zip_nome = list(uploaded.keys())[0]
pasta_destino = "dados_extraidos"

with zipfile.ZipFile(zip_nome, 'r') as zip_ref:
    zip_ref.extractall(pasta_destino)

print(f"Arquivo '{zip_nome}' extraído com sucesso em '{pasta_destino}'!")

Selecione o arquivo 'size_normalized_texts.zip' do seu computador:


Saving size_normalized_texts.zip to size_normalized_texts.zip
Arquivo 'size_normalized_texts.zip' extraído com sucesso em 'dados_extraidos'!


In [2]:
import glob
import pandas as pd

textos = []
labels = []

# Varredura recursiva de arquivos .txt nas pastas fake e true
arquivos_fake = glob.glob(f"{pasta_destino}/**/fake/*.txt", recursive=True)
arquivos_true = glob.glob(f"{pasta_destino}/**/true/*.txt", recursive=True)

# 1 = FALSA
for cam in arquivos_fake:
    try:
        with open(cam, 'r', encoding='utf-8', errors='ignore') as f:
            conteudo = f.read().strip()
            if conteudo:
                textos.append(conteudo)
                labels.append(1)
    except Exception:
        pass

# 0 = VERDADEIRA
for cam in arquivos_true:
    try:
        with open(cam, 'r', encoding='utf-8', errors='ignore') as f:
            conteudo = f.read().strip()
            if conteudo:
                textos.append(conteudo)
                labels.append(0)
    except Exception:
        pass

df = pd.DataFrame({'texto': textos, 'label': labels})

print(f"Total de registros carregados: {len(df)}")
print("Distribuição das classes:")
print(f"- Falsas (1): {(df['label'] == 1).sum()}")
print(f"- Verdadeiras (0): {(df['label'] == 0).sum()}")

Total de registros carregados: 1821
Distribuição das classes:
- Falsas (1): 1000
- Verdadeiras (0): 821


In [3]:
from sklearn.model_selection import train_test_split

# Separação estrita de 30% para teste preservado (547 amostras)
# O conjunto de teste permanece intocado e fora do balanceamento por clustering
X_treino_bruto, X_teste, y_treino_bruto, y_teste = train_test_split(
    df['texto'],
    df['label'],
    test_size=0.30,
    random_state=42,
    stratify=df['label']
)

print(f"Treino inicial: {len(X_treino_bruto)} amostras")
print(f"Teste preservado: {len(X_teste)} amostras")

Treino inicial: 1274 amostras
Teste preservado: 547 amostras


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans

df_treino = pd.DataFrame({'texto': X_treino_bruto, 'label': y_treino_bruto})

# TF-IDF temático para delimitação dos clusters
tfidf_cluster = TfidfVectorizer(min_df=3, max_features=50000)
X_cluster_matriz = tfidf_cluster.fit_transform(df_treino['texto'])

# K-Means com 30 clusters temáticos
kmeans = MiniBatchKMeans(n_clusters=30, batch_size=1024, random_state=42, n_init=3)
df_treino['cluster'] = kmeans.fit_predict(X_cluster_matriz)

df_verdadeiras = df_treino[df_treino['label'] == 0]
df_falsas = df_treino[df_treino['label'] == 1]
n_verdadeiras = len(df_verdadeiras)

# Subamostragem por cotas entre os clusters para mitigação de Topic Bias
amostras_falsas = []
cota_por_cluster = max(1, n_verdadeiras // 30)

for c in range(30):
    falsas_cluster = df_falsas[df_falsas['cluster'] == c]
    n_pegar = min(len(falsas_cluster), cota_por_cluster)
    if n_pegar > 0:
        amostras_falsas.append(falsas_cluster.sample(n=n_pegar, random_state=42))

df_falsas_amostradas = pd.concat(amostras_falsas)

# Ajuste fino para paridade exata
faltantes = n_verdadeiras - len(df_falsas_amostradas)
if faltantes > 0:
    restante = df_falsas.drop(df_falsas_amostradas.index)
    df_falsas_amostradas = pd.concat([df_falsas_amostradas, restante.sample(n=faltantes, random_state=42)])

# Consolidação do conjunto de treino balanceado
df_treino_balanceado = pd.concat([df_verdadeiras, df_falsas_amostradas]).sample(frac=1, random_state=42)

X_treino_final = df_treino_balanceado['texto']
y_treino_final = df_treino_balanceado['label']

print(f"Treino balanceado: {len(X_treino_final)} amostras")
print(f"Verdadeiras: {(y_treino_final == 0).sum()} | Falsas: {(y_treino_final == 1).sum()}")

Treino balanceado: 1148 amostras
Verdadeiras: 574 | Falsas: 574


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
import joblib

# 1. Pipeline Estilométrico (Unigramas e Bigramas)
vetorizador = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=150000)
X_treino_tfidf = vetorizador.fit_transform(X_treino_final)

# 2. Regressão Logística com pesos balanceados
modelo_base = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# 3. Calibração de Platt via Sigmoide
modelo_calibrado = CalibratedClassifierCV(estimator=modelo_base, method='sigmoid', cv=5)
modelo_calibrado.fit(X_treino_tfidf, y_treino_final)

# 4. Exportação do artefato unificado
artefato = {
    'vetorizador': vetorizador,
    'modelo': modelo_calibrado
}

nome_arquivo_pkl = "modelo_classificador_fakebr.pkl"
joblib.dump(artefato, nome_arquivo_pkl)
print(f"Modelo salvo em '{nome_arquivo_pkl}' com sucesso.")

# Download direto para o seu computador
files.download(nome_arquivo_pkl)

Modelo salvo em 'modelo_classificador_fakebr.pkl' com sucesso.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Transformação do conjunto preservado
X_teste_tfidf = vetorizador.transform(X_teste)

# Predição
y_pred = modelo_calibrado.predict(X_teste_tfidf)
y_proba = modelo_calibrado.predict_proba(X_teste_tfidf)

print("=" * 60)
print("RELATÓRIO DE DESEMPENHO NO CONJUNTO DE TESTE (547 AMOSTRAS)")
print("=" * 60)
print(classification_report(y_teste, y_pred, target_names=["Verdadeira (0)", "Falsa (1)"]))

print("MATRIZ DE CONFUSÃO:")
print(confusion_matrix(y_teste, y_pred))

print(f"\nAcurácia Geral: {accuracy_score(y_teste, y_pred):.2%}")
print(f"F1-Score Macro: {f1_score(y_teste, y_pred, average='macro'):.4f}")

RELATÓRIO DE DESEMPENHO NO CONJUNTO DE TESTE (547 AMOSTRAS)
                precision    recall  f1-score   support

Verdadeira (0)       0.91      0.91      0.91       247
     Falsa (1)       0.93      0.93      0.93       300

      accuracy                           0.92       547
     macro avg       0.92      0.92      0.92       547
  weighted avg       0.92      0.92      0.92       547

MATRIZ DE CONFUSÃO:
[[225  22]
 [ 22 278]]

Acurácia Geral: 91.96%
F1-Score Macro: 0.9188


In [7]:
# Teste de carregamento a frio do arquivo .pkl gerado
artefato_carregado = joblib.load(nome_arquivo_pkl)
vec_carregado = artefato_carregado['vetorizador']
mod_carregado = artefato_carregado['modelo']

def inferir_com_abstencao(texto, limiar=0.65):
    vec = vec_carregado.transform([texto])
    p_verdadeira, p_falsa = mod_carregado.predict_proba(vec)[0]

    if max(p_verdadeira, p_falsa) < limiar:
        classificacao = "INCERTO (Abstenção por margem ambígua)"
    elif p_falsa > p_verdadeira:
        classificacao = "FALSA"
    else:
        classificacao = "VERDADEIRA"

    return {
        "resultado": classificacao,
        "prob_verdadeira": f"{p_verdadeira:.2%}",
        "prob_falsa": f"{p_falsa:.2%}"
    }

exemplo_teste = "O plenário do Supremo Tribunal Federal concluiu o julgamento conjunto na tarde desta quinta-feira."
resultado = inferir_com_abstencao(exemplo_teste)
print("Teste de Inferência:")
print(resultado)

Teste de Inferência:
{'resultado': 'VERDADEIRA', 'prob_verdadeira': '99.97%', 'prob_falsa': '0.03%'}
